<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 4–5 扩展实验：文件直查、批量落地与合批确认</h1>
<p>目标 Doris 4.1.3 · 独立 ext_* 实验表</p>
</div>

[扩展入口](README.md) · [主线学习目录](../README.md)

先完成 Lab 1、4、5，使用相同课程沙箱与 MinIO。此实验会确认并准备课程湖表环境，不连接外部生产服务。
仅重建 `ext_file_insert`、`ext_file_broker`、`ext_defaults` 和 `ext_gc_off_mode/sync_mode/async_mode`。
对象存储只新增当前实验库下的随机文件路径，不删除其他文件。建议 30–45 分钟，独立于主线 Lab。
一次只运行一个内核，失败先查看原始导入响应及 SHOW LOAD；不要直接重发异步导入。

验收：同一 WWI 十单文件直查与两种落地结果逐行一致；默认值/生成列可核对；三种确认模式最终均为 10 行、1400.00。
这不是 Kafka、CDC 或持续文件发现实验，也不验证 WAL 磁盘故障恢复。

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. 把十单样本写成独立 Parquet 文件

这里创建的是独立文件，不是 Iceberg 表。文件与 Lab 4 湖表使用同一十单样本。
Python 访问宿主机 51900；Doris 通过课程网络访问 course-lake-minio:9000，二者不要混用。

In [ ]:
from dw_course.lakehouse import prepare_lakehouse, ACCESS_KEY, SECRET_KEY
from dw_course.wwi import history_rows, history_ddl, HISTORY_COLUMNS
from decimal import Decimal
from datetime import date
from uuid import uuid4
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
lake_table = prepare_lakehouse(lab, start=True)
records = [dict(zip(HISTORY_COLUMNS, row)) for row in history_rows()]
for row in records:
    row["order_amount"] = Decimal(row["order_amount"])
    row["order_date"] = date.fromisoformat(row["order_date"])
buffer = pa.BufferOutputStream()
pq.write_table(pa.Table.from_pylist(records), buffer)
key = f"{lab.database}/file_demo/{uuid4().hex}/orders.parquet"
s3 = boto3.client("s3",endpoint_url="http://127.0.0.1:51900",aws_access_key_id=ACCESS_KEY,aws_secret_access_key=SECRET_KEY,region_name="us-east-1")
s3.put_object(Bucket="course-warehouse",Key=key,Body=buffer.getvalue().to_pybytes())
uri = "s3://course-warehouse/" + key
tvf = f'''S3("uri"="{uri}","format"="parquet",
"s3.endpoint"="http://course-lake-minio:9000","s3.region"="us-east-1",
"s3.access_key"="{ACCESS_KEY}","s3.secret_key"="{SECRET_KEY}","use_path_style"="true")'''
projection = "order_id,customer_id,CAST(order_date AS STRING),order_amount,line_count,data_source"
expect(lab.query(f"SELECT {projection} FROM {tvf} ORDER BY order_id"), history_rows())
expect(lab.query(f"SELECT {projection} FROM {lake_table} ORDER BY order_id"), history_rows())
lab.sql(f"SELECT COUNT(*) AS orders,SUM(order_amount) AS amount FROM {tvf}")

## 2. 同步 INSERT SELECT 与异步 Broker Load

两种方法写入独立表。`WITH S3` 使用 S3 协议，不需要额外 Broker 进程。
Broker Load 提交成功后必须等 `State=FINISHED`；取消或超时即失败，不能继续输出通过。
超时会请求取消本次唯一 label，不重置整个环境。失败详情保留在 SHOW LOAD 中。

In [ ]:
import time
for table in ("ext_file_insert", "ext_file_broker"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(history_ddl(table))
lab.execute(f"INSERT INTO ext_file_insert ({','.join(HISTORY_COLUMNS)}) SELECT {','.join(HISTORY_COLUMNS)} FROM {tvf}")
label = "ext_broker_" + uuid4().hex
statement = f'''LOAD LABEL {lab.database}.{label} (
DATA INFILE("{uri}") INTO TABLE ext_file_broker FORMAT AS "parquet"
({','.join(HISTORY_COLUMNS)})
) WITH S3 (
"AWS_ENDPOINT"="http://course-lake-minio:9000","AWS_REGION"="us-east-1",
"AWS_ACCESS_KEY"="{ACCESS_KEY}","AWS_SECRET_KEY"="{SECRET_KEY}","use_path_style"="true"
) PROPERTIES ("timeout"="120","max_filter_ratio"="0")'''
show_sql("提交批量任务", statement)
lab.execute(statement)
deadline = time.monotonic()+150
while True:
    rows = lab.query("SHOW LOAD WHERE Label = %s", (label,))
    if rows and rows[0][2] == "FINISHED":
        break
    if rows and rows[0][2] == "CANCELLED":
        raise RuntimeError(f"Broker Load cancelled: {rows}")
    if time.monotonic() >= deadline:
        lab.execute("CANCEL LOAD WHERE LABEL = %s", (label,))
        raise TimeoutError(f"Broker Load timeout: {rows}")
    time.sleep(1)
lab.sql("SHOW LOAD WHERE Label = %s", (label,))
for table in ("ext_file_insert", "ext_file_broker"):
    expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), history_rows())

## 3. 默认值和生成列不是同一件事

省略 status 时填入 CREATED；显式指定 PAID 时保留输入。
amount 根据 amount_cents 生成：18000 分 → 180.00 元。INSERT 不提供生成列。
这里的两笔是独立教学样本，不追加到历史或当前状态表。

In [ ]:
lab.execute("DROP TABLE IF EXISTS ext_defaults")
lab.execute('''CREATE TABLE ext_defaults (
order_id BIGINT, amount_cents BIGINT NOT NULL,
status VARCHAR(20) NOT NULL DEFAULT "CREATED",
amount DECIMAL(12,2) AS (CAST(amount_cents AS DECIMAL(12,2)) / 100)
) DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents) VALUES (901001,18000)")
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents,status) VALUES (901002,8000,'PAID')")
expect(lab.query("SELECT order_id,status,amount FROM ext_defaults ORDER BY order_id"), [(901001,"CREATED","180.00"),(901002,"PAID","80.00")])
lab.sql("SELECT * FROM ext_defaults ORDER BY order_id")

## 4. 合批响应时间不等于查询可见时间

分别向三张空表发送相同十笔模拟订单，记录收到响应和首次查询到全部数据的时间。
合批请求不自定义 label；以返回 GroupCommit 字段确认路径，不把 off_mode 的 label 重试结论套过来。
观察 async 的首次查询是否已经可见，不强制期待“必定不可见”，也不固定延迟倍数。
本实验只比较确认与可见性；多个请求是否合并为同一事务、持续吞吐与版本积压需另做并发负载实验。

In [ ]:
import os
import requests
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
payload = (COURSE_ROOT / "datasets/orders.csv").read_bytes()
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
for mode in ("off_mode", "sync_mode", "async_mode"):
    table = "ext_gc_" + mode
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(order_ddl(table))
    lab.execute(f'ALTER TABLE {table} SET ("group_commit_interval_ms"="2000")')
    headers = {"format":"csv", "column_separator":",", "columns":",".join(ORDER_COLUMNS),
               "strict_mode":"true", "max_filter_ratio":"0", "group_commit":mode}
    if mode == "off_mode":
        headers["label"] = "ext_off_" + uuid4().hex
    started = time.monotonic()
    response = requests.put(f"{endpoint}/api/{lab.database}/{table}/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload,
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    print(mode,result)
    expect(result["Status"], "Success")
    expect(result["NumberLoadedRows"], 10)
    if mode != "off_mode":
        expect(result["GroupCommit"], True)
    acknowledged = time.monotonic()
    first_count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    count = first_count
    while count != 10:
        if count > 10:
            raise RuntimeError(f"Unexpected duplicate rows: {count}")
        if time.monotonic()-acknowledged > 60:
            raise TimeoutError(f"Data not visible: {result}")
        time.sleep(0.1)
        count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    visible = time.monotonic()
    projection = ",".join("DATE_FORMAT(event_time,'%Y-%m-%d %H:%i:%s')" if col=="event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), order_rows(fixture("orders.json")))
    print(mode, "响应ms",round((acknowledged-started)*1000,2), "首次完整可见ms",round((visible-started)*1000,2),"响应后首查行数",first_count)
    lab.sql(f"SHOW TABLETS FROM {table}")

## 5. 独立解释与排查

解释为什么文件直查不用 CREATE CATALOG，而 Iceberg 需要；为什么 LOAD LABEL 返回后不能立刻算完成。
再解释默认值与生成列各在哪一步生效，为什么 async 成功不能单靠响应证明已可查。
失败时保留 response、label、TxnId 与 SHOW LOAD；Group Commit 超时先排查 WAL/任务，不能盲目重发。

参考：[S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/)、
[Broker Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/broker-load-manual/)、
[Group Commit](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/group-commit-manual/)、
[CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)。

In [ ]:
lab.close()